# Checkpointing & Persistence

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will learn how to use `InMemorySaver` for checkpointing, `thread_id` for session isolation, `get_state()` and `get_state_history()` for inspection, and how to build a multi-turn chatbot with persistent memory.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Multi-Turn Chatbot with InMemorySaver

Build a chatbot that remembers messages across invocations using `InMemorySaver` and `thread_id`.

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(MessagesState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

checkpointer = InMemorySaver()
app = graph.compile(checkpointer=checkpointer)

In [ ]:
config = {"configurable": {"thread_id": "demo"}}

result1 = app.invoke({"messages": [("human", "My name is Alice")]}, config=config)
print(f"Turn 1: {result1['messages'][-1].content}")

result2 = app.invoke({"messages": [("human", "What is my name?")]}, config=config)
print(f"Turn 2: {result2['messages'][-1].content}")

## 4. Thread Isolation

Different `thread_id` values create independent conversations.

In [ ]:
config_a = {"configurable": {"thread_id": "user-alice"}}
config_b = {"configurable": {"thread_id": "user-bob"}}

app.invoke({"messages": [("human", "My favorite color is blue")]}, config=config_a)
app.invoke({"messages": [("human", "My favorite color is red")]}, config=config_b)

result_a = app.invoke({"messages": [("human", "What is my favorite color?")]}, config=config_a)
print(f"Alice's thread: {result_a['messages'][-1].content}")

result_b = app.invoke({"messages": [("human", "What is my favorite color?")]}, config=config_b)
print(f"Bob's thread: {result_b['messages'][-1].content}")

## 5. Inspecting State with get_state()

`get_state()` returns a `StateSnapshot` with the current values, next nodes, and metadata.

In [ ]:
snapshot = app.get_state(config)

print(f"Number of messages: {len(snapshot.values['messages'])}")
print(f"Next nodes: {snapshot.next}")
print(f"Created at: {snapshot.created_at}")
print(f"Step: {snapshot.metadata['step']}")

## 6. Walking History with get_state_history()

Iterate through every checkpoint saved for a thread.

In [ ]:
for snapshot in app.get_state_history(config):
    print(f"Step {snapshot.metadata['step']}: next={snapshot.next}, messages={len(snapshot.values['messages'])}")

## 7. StateSnapshot Fields Reference

| Field | Description |
|-------|-------------|
| `values` | The full state dict at this checkpoint |
| `next` | Tuple of node names that would execute next |
| `config` | Config dict including `thread_id` and `checkpoint_id` |
| `parent_config` | Config pointing to the previous checkpoint |
| `metadata` | Dict with `source`, `step`, `writes`, `parents` |
| `created_at` | ISO timestamp of checkpoint creation |
| `tasks` | Pending tasks at this checkpoint |

## What You Learned

1. **`InMemorySaver`** enables checkpointing for development and testing
2. **`thread_id`** isolates independent conversations within the same graph
3. **`get_state(config)`** returns a `StateSnapshot` with current values and metadata
4. **`get_state_history(config)`** yields every checkpoint for time-travel debugging
5. Multi-turn chatbots are trivial with persistence — just reuse the same `thread_id`